In [0]:
from typing import Any

In [0]:
import logging

from py4j.java_gateway import JavaObject  # type: ignore
from pyspark.sql import SparkSession
from pyspark.sql.functions import (
    concat_ws,
    current_timestamp,
    from_utc_timestamp,
    lit,
    sha2,
    when,
)

In [0]:
dbutils.widgets.text('curated_zone', 'None', '01-Curated_zone')
dbutils.widgets.text('catalog_landed', 'None', '02-Catalog_landed')
dbutils.widgets.text('reference_zone', 'None', '03-Reference_zone')
dbutils.widgets.text('run_date', 'None', '04-Run Date')
dbutils.widgets.text('schema', 'None', '05-Schema')
dbutils.widgets.text('env_prefix', 'None', '06-Environment')
curated_zone = dbutils.widgets.get('curated_zone')
catalog_landed = dbutils.widgets.get('catalog_landed')
reference_zone = dbutils.widgets.get('reference_zone')
run_date = dbutils.widgets.get('run_date')
schema = dbutils.widgets.get('schema')
env_prefix = dbutils.widgets.get('env_prefix')

import sys, os
relative_path = '../../helpers/lng_lnd'
path = os.path.abspath(relative_path)
if path not in sys.path:
  sys.path.append(path)
from landing_helpers import *
from landed_helpers import *
relative_path = '../../helpers/curated_helpers'
path = os.path.abspath(relative_path)
if path not in sys.path:
  sys.path.append(path)
from delta_table_creator import *
from extract import *
from transform import *
spark = SparkSession.builder.getOrCreate()


In [0]:
def load_curated_real_estate_valuation_arotron(
    spark, curated_zone, catalog_landed, reference_zone, run_date, schema, env_prefix
) -> None:
    args = [curated_zone, catalog_landed, reference_zone, run_date, schema, env_prefix]
    if all(item is not None for item in args) :
        date_filter = f"businessdate = {run_date!r}"

        # schema = (
        #     "RealEstateValuationId String, SourceSystem string, PortfolioKey string, PortfolioName string, IngestionTimestamp timestamp,"
        #     "BatchId string, RealEstateValuationKey string, RealEstateKey string, RealEstateDescription string, RealEstateTypeCode string,"
        #     "RealEstateType string, RealEstateValuationStatusCode string, RealEstateValuationStatus string, RealEstateValuatorKey string,"
        #     "RealEstateValuationType1 string, RealEstateValuationType2 string, ValuationPurpose string, ValuationMethod string,"
        #     "ValuationMethodWeights string, MarketabilityIndex string, AutopsyScope string, AutopsyDate timestamp, ValuationDate timestamp,"
        #     "MarketValue Decimal(19,4), LeaseValue Decimal(19,4), InsuranceValue Decimal(19,4), ReasonableValue Decimal(19,4),"
        #     "FiscalValue Decimal(19,4), ReplacementValue Decimal(19,4), EquipmentValue Decimal(19,4), ValuationCost Decimal(19,4),"
        #     "OtherCost Decimal(19,4), Address1 string, Address2 string, ZipCode string, City string, Area string, Geolocation string,"
        #     "DetectionProblemFlag boolean, AreaTopographic double, AreaTitle double, AreaComments string,"
        #     "BuildingFactor double, FeasibleFactor double, CoverageFactor double, AreaEven int, AreaMin int,"
        #     "FacadeMin int, MaxHeight double, BuildingIntroDescription string, BuildingPermit string,"
        #     "BuildingReadiness double, CompletionYear int, Comments string, EnteredBy int, DateEntered timestamp, TimeToSell int,"
        #     "IsActiveFlag boolean, ActionCode int, HtmlRealTitle string, HtmlDocuments string, HtmlLand string, HtmlLimitations string,"
        #     "HtmlAlloweduse string, HtmlDeclaration string, HtmlBuildingurban string, HtmlBuildinglegal string, HtmlAssignor string,"
        #     "HtmlAttention string, HtmlOwnershipHeader string, HtmlOwnershipFooter string, BusinessDate date"
        # )

        cluster_columns = ["BusinessDate"]

        curated_table_path = f"{curated_zone}.RealEstateValuation"

        # ================== Create Delta Table ==================
        logging.info("Creating Delta Table...")
        if spark.catalog.tableExists(curated_table_path) is False:
            create_delta_table(
                spark,
                curated_table_path,
                cluster_columns,
                schema,  # type: ignore
            )
        else:
            logging.info("Table already exists.")

        # ================== Delete From Delta Table ==================
        if spark.catalog.tableExists(curated_table_path) is True:
            delete_from_curated_table(
                spark,
                curated_table_path,
                run_date,  # type: ignore
                "arotron",
            )        
        else:
            logging.info("Tables does not exist, no need to delete entries for business date!")

        # ================== eaiSIG_REPE ==================

        eaiSIG_REPE_extra_filters = None

        if eaiSIG_REPE_extra_filters is not None:
            eaiSIG_REPE_extra_filters.append(date_filter)
            eaiSIG_REPE_final_filters = " AND ".join(eaiSIG_REPE_extra_filters)
        else:
            eaiSIG_REPE_final_filters = date_filter

        eaiSIG_REPE_selected_fields = [
            "eaiSIG_REPE_EVAL_CODE",
            "eaiSIG_REPE_REAL_ESTATE_CODE",
            "eaiSIG_REPE_REST_APPR_SOURCE",
            "eaiSIG_REPE_REST_EVALUATION_DATE",
            "eaiSIG_REPE_REST_MARKET_VALUE",
            "eaiSIG_REPE_REST_INSURANCE_VALUE",
            "eaiSIG_REPE_REST_COMMERCIAL",
            "eaiSIG_REPE_REST_OBJECTIVE_DATE",
            "businessdate",
        ]

        eaiSIG_REPE = load_landed_delta_table(
            spark,
            catalog_landed,  # type: ignore
            "arotron_eaiSIG_REPE",
            eaiSIG_REPE_final_filters,
            eaiSIG_REPE_selected_fields,
        )

        # ================== eaiSIG_RE ==================

        eaiSIG_RE_extra_filters = None

        if eaiSIG_RE_extra_filters is not None:
            eaiSIG_RE_extra_filters.append(date_filter)
            eaiSIG_RE_final_filters = " AND ".join(eaiSIG_RE_extra_filters)
        else:
            eaiSIG_RE_final_filters = date_filter

        eaiSIG_RE_selected_fields = [
            "eaiSIG_RE_REAL_ESTATE_CODE",
            "eaiSIG_RE_ADDRESS",
            "eaiSIG_RE_DISTRICT",
            "eaiSIG_RE_TOPONYM",
            "eaiSIG_RE_POSTAL_CODE",
            "eaiSIG_RE_COMMENTS",
            "businessdate",
        ]

        eaiSIG_RE = load_landed_delta_table(  # noqa: F841
            spark,
            catalog_landed,  # type: ignore
            "arotron_eaiSIG_RE",
            eaiSIG_RE_final_filters,
            eaiSIG_RE_selected_fields,
        )

        eaiSIG_RE_eaiSIG_REPE_join_condition = (
            eaiSIG_RE["eaiSIG_RE_REAL_ESTATE_CODE"]
            == eaiSIG_REPE["eaiSIG_REPE_REAL_ESTATE_CODE"]
        )

        # Joins
        target_table_arotron = eaiSIG_REPE.join(
            eaiSIG_RE, eaiSIG_RE_eaiSIG_REPE_join_condition, "LEFT"
        )

        # HashKeys
        target_table_arotron_RealEstateValuationId_primary_key_fields = [
            eaiSIG_REPE.eaiSIG_REPE_EVAL_CODE,
            eaiSIG_REPE.eaiSIG_REPE_REAL_ESTATE_CODE,
            lit("arotron"),
        ]

        target_table_arotron = (
            target_table_arotron.withColumn(
                "RealEstateValuationId",
                sha2(
                    concat_ws(
                        ";",
                        *target_table_arotron_RealEstateValuationId_primary_key_fields,
                    ),
                    256,
                ),
            )
            .withColumn(
                "RealEstateValuationKey",
                concat_ws(
                    ";",
                    eaiSIG_REPE.eaiSIG_REPE_EVAL_CODE,
                    eaiSIG_REPE.eaiSIG_REPE_REAL_ESTATE_CODE,
                ),
            )
            .withColumn(
                "Address2",
                when(
                    eaiSIG_RE.eaiSIG_RE_DISTRICT != eaiSIG_RE.eaiSIG_RE_TOPONYM,
                    concat_ws(
                        ";", eaiSIG_RE.eaiSIG_RE_DISTRICT, eaiSIG_RE.eaiSIG_RE_TOPONYM
                    ),
                ).otherwise(eaiSIG_RE.eaiSIG_RE_DISTRICT),
            )
            .withColumn(
                "ValuationDate",
                when(
                    eaiSIG_REPE.eaiSIG_REPE_REST_EVALUATION_DATE == "", lit(None)
                ).otherwise(
                    eaiSIG_REPE.eaiSIG_REPE_REST_EVALUATION_DATE.cast("timestamp")
                ),
            )
            .withColumn(
                "MarketValue",
                when(eaiSIG_REPE.eaiSIG_REPE_REST_MARKET_VALUE == "", -9999).otherwise(
                    eaiSIG_REPE.eaiSIG_REPE_REST_MARKET_VALUE.cast("decimal(19,4)")
                ),
            )
            .withColumn(
                "InsuranceValue",
                when(
                    eaiSIG_REPE.eaiSIG_REPE_REST_INSURANCE_VALUE == "", lit(None)
                ).otherwise(
                    eaiSIG_REPE.eaiSIG_REPE_REST_INSURANCE_VALUE.cast("decimal(19,4)")
                ),
            )
            .withColumn(
                "ReasonableValue",
                when(
                    eaiSIG_REPE.eaiSIG_REPE_REST_COMMERCIAL == "", lit(None)
                ).otherwise(
                    eaiSIG_REPE.eaiSIG_REPE_REST_COMMERCIAL.cast("decimal(19,4)")
                ),
            )
            .withColumn(
                "FiscalValue",
                when(
                    eaiSIG_REPE.eaiSIG_REPE_REST_OBJECTIVE_DATE == "", lit(None)
                ).otherwise(
                    eaiSIG_REPE.eaiSIG_REPE_REST_OBJECTIVE_DATE.cast("decimal(19,4)")
                ),
            )
        )

        # ================== Selection of Final Columns ==================

        result_df_arotron = target_table_arotron.select(
            target_table_arotron.RealEstateValuationId.alias("RealEstateValuationId"),
            lit("arotron").cast("String").alias("SourceSystem"),
            lit(-9998).cast("string").alias("PortfolioKey"),
            lit("Core Wholesale").cast("String").alias("PortfolioName"),
            lit(from_utc_timestamp(current_timestamp(), "Europe/Athens"))
            .cast("timestamp")
            .alias("IngestionTimestamp"),
            lit("test").cast("String").alias("BatchId"),
            target_table_arotron.RealEstateValuationKey.cast("string").alias(
                "RealEstateValuationKey"
            ),
            target_table_arotron.eaiSIG_REPE_REAL_ESTATE_CODE.cast("string").alias(
                "RealEstateKey"
            ),
            lit(None).cast("String").alias("RealEstateDescription"),
            lit(None).cast("String").alias("RealEstateTypeCode"),
            lit(None).cast("String").alias("RealEstateType"),
            lit(None).cast("String").alias("RealEstateValuationStatusCode"),
            lit(None).cast("String").alias("RealEstateValuationStatus"),
            lit(None).cast("string").alias("RealEstateValuatorKey"),
            eaiSIG_REPE.eaiSIG_REPE_REST_APPR_SOURCE.cast("String").alias(
                "RealEstateValuationType1"
            ),
            lit(None).cast("String").alias("RealEstateValuationType2"),
            lit(None).cast("String").alias("ValuationPurpose"),
            eaiSIG_REPE.eaiSIG_REPE_REST_APPR_SOURCE.cast("String").alias(
                "ValuationMethod"
            ),
            lit(None).cast("String").alias("ValuationMethodWeights"),
            lit(None).cast("String").alias("MarketabilityIndex"),
            lit(None).cast("String").alias("AutopsyScope"),
            lit(None).cast("timestamp").alias("AutopsyDate"),
            target_table_arotron.ValuationDate.alias(
                "ValuationDate"
            ),  # problem, the field is actually of the "String" data type for the arotron source system. The convertion to timestamp works but it results in values that don't make sense. The blank values get converted to 0x0000000000000000.
            target_table_arotron.MarketValue.alias(
                "MarketValue"
            ),  # problem, the field is actually of the "String" data type for the arotron source system, but the convertion to decimal(19,4) works for non-blank values. For the blank values this doesn't work.
            lit(None).cast("decimal(19,4)").alias("LeaseValue"),
            target_table_arotron.InsuranceValue.alias(
                "InsuranceValue"
            ),  # problem, the field is actually of the "String" data type for the arotron source system, but the convertion to decimal(19,4) works for non-blank values. For the blank values this doesn't work.
            target_table_arotron.ReasonableValue.alias(
                "ReasonableValue"
            ),  # problem, the field is actually of the "String" data type for the arotron source system, but the convertion to decimal(19,4) works for non-blank values. For the blank values this doesn't work.
            target_table_arotron.FiscalValue.alias(
                "FiscalValue"
            ),  # problem, the field is actually of the "String" data type for the arotron source system, but the convertion to decimal(19,4) works for non-blank values. For the blank values this doesn't work.
            lit(None).cast("decimal(19,4)").alias("ReplacementValue"),
            lit(None).cast("decimal(19,4)").alias("EquipmentValue"),
            lit(None).cast("decimal(19,4)").alias("ValuationCost"),
            lit(None).cast("decimal(19,4)").alias("OtherCost"),
            eaiSIG_RE.eaiSIG_RE_ADDRESS.cast("String").alias("Address1"),
            target_table_arotron.Address2.alias("Address2"),
            eaiSIG_RE.eaiSIG_RE_POSTAL_CODE.cast("String").alias("ZipCode"),
            lit(None).cast("String").alias("City"),
            lit(None).cast("String").alias("Area"),
            lit(None).cast("String").alias("Geolocation"),
            lit(None).cast("boolean").alias("DetectionProblemFlag"),
            lit(None).cast("double").alias("AreaTopographic"),
            lit(None).cast("double").alias("AreaTitle"),
            lit(None).cast("String").alias("AreaComments"),
            lit(None).cast("double").alias("BuildingFactor"),
            lit(None).cast("double").alias("FeasibleFactor"),
            lit(None).cast("double").alias("CoverageFactor"),
            lit(None).cast("int").alias("AreaEven"),
            lit(None).cast("int").alias("AreaMin"),
            lit(None).cast("int").alias("FacadeMin"),
            lit(None).cast("double").alias("MaxHeight"),
            lit(None).cast("String").alias("BuildingIntroDescription"),
            lit(None).cast("String").alias("BuildingPermit"),
            lit(None).cast("double").alias("BuildingReadiness"),
            lit(None).cast("int").alias("CompletionYear"),
            eaiSIG_RE.eaiSIG_RE_COMMENTS.cast("String").alias("Comments"),
            lit(None).cast("int").alias("EnteredBy"),
            lit(None).cast("timestamp").alias("DateEntered"),
            lit(None).cast("int").alias("TimeToSell"),
            lit(None).cast("boolean").alias("IsActiveFlag"),
            lit(None).cast("int").alias("ActionCode"),
            lit(None).cast("String").alias("HtmlRealTitle"),
            lit(None).cast("String").alias("HtmlDocuments"),
            lit(None).cast("String").alias("HtmlLand"),
            lit(None).cast("String").alias("HtmlLimitations"),
            lit(None).cast("String").alias("HtmlAlloweduse"),
            lit(None).cast("String").alias("HtmlDeclaration"),
            lit(None).cast("String").alias("HtmlBuildingurban"),
            lit(None).cast("String").alias("HtmlBuildinglegal"),
            lit(None).cast("String").alias("HtmlAssignor"),
            lit(None).cast("String").alias("HtmlAttention"),
            lit(None).cast("String").alias("HtmlOwnershipHeader"),
            lit(None).cast("String").alias("HtmlOwnershipFooter"),
            eaiSIG_REPE.businessdate.cast("date").alias("BusinessDate"),
        )

        result_df_arotron.write.format("delta").clusterBy("BusinessDate").mode(
            "append"
        ).saveAsTable(curated_table_path)

    else:
        null_args = [
            arg_name
            for arg_name, arg in args
            if arg
            in [
                curated_zone,
                catalog_landed,
                reference_zone,
                run_date,
                schema,
                env_prefix,
            ]
            and arg is None
        ]

        for arg_name in null_args:
            logging.error(f"No {arg_name} specified!")

    logging.info("We did it!")


In [0]:
load_curated_real_estate_valuation_arotron(spark, curated_zone, catalog_landed, reference_zone, run_date, schema, env_prefix)
